Pipeline overview:
1. For each emotion, load every subject's height x width body map from .npz files (key: "resmat").
2. Run a two-sided one-sample cluster permutation test (sign-flipping) via MNE.
3. Save per-emotion results (t map, significant masks) and a CSV of significant clusters.
4. Plot group-average maps with significant clusters outlined (red = positive, blue = negative).

In [ ]:
from pathlib import Path
import csv
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from scipy.stats import t as t_distribution
from datetime import datetime
from mne.stats import permutation_cluster_1samp_test
from scipy.sparse import coo_matrix
import subprocess

def load_one_emotion(npz_folder, emotion_index,height,width):
    """
    Load a single emotion's map for every subject in npz_folder.

    Only the requested slice of each subject's resmat is pulled into
    memory - the rest of the array is discarded immediately - so this
    never holds more than one emotion x all subjects in RAM at once.

    Parameters
    ----------
    npz_folder : str or Path
    emotion_index : int
        Index into the last axis of each subject's resmat array.

    Returns
    -------
    subject_maps : ndarray, shape (n_subjects, height, width)
    """
    npz_folder = Path(npz_folder)
    subject_maps = []

    for file_path in sorted(npz_folder.glob("*.npz")):
        with np.load(file_path) as saved:
            if "resmat" not in saved:
                print(f"Skipping {file_path.name}: no 'resmat' variable")
                continue
            # slice just this emotion, cast to float32 -> discard the rest immediately
            # NOTE: saved["resmat"] reads the full array from disk before slicing, so each
            # file is re-read once per emotion (16x total). Peak memory stays low, I/O doesn't.
            emotion_map = saved["resmat"][:, :, emotion_index].astype(np.float32)

        # Guards against subjects saved with a different crop/resolution.
        if emotion_map.shape != (height, width):
            print(
                f"Skipping {file_path.name}: "
                f"shape {emotion_map.shape}, expected ({height}, {width})")
            continue

        subject_maps.append(emotion_map)

    print(f"{len(subject_maps)} subjects loaded")

    return np.stack(subject_maps)


def safe_t_statistic(X):
    # X: observations x features. MNE flattens the (subjects, 600, 235) input to
    # (subjects, 141000) before calling this, and calls it once per permutation.
    X = np.asarray(X, dtype=float)
    # Standard error of the mean, per pixel (ddof=1 = sample variance).
    se = np.sqrt(X.var(axis=0, ddof=1) / X.shape[0])
    # Preallocated zeros: np.divide with where= leaves other positions untouched,
    # so constant pixels (se == 0) come out as t = 0 instead of inf/NaN or garbage.
    t = np.zeros(X.shape[1])
    # A non-finite mean would also make se non-finite, so the se check covers both.
    np.divide(X.mean(axis=0), se, out=t, where=np.isfinite(se) & (se > 0))
    return t

def create_8_connectivity(h, w):
    """
        Create 8-neighbor connectivity for a 2D image.

        Each pixel is connected to:
            up, down, left, right,
            and the four diagonal neighbors.
        """
    idx = np.arange(h * w).reshape(h, w)
    rows, cols = [], []
    for dr in (-1, 0, 1):
        for dc in (-1, 0, 1):
            if dr == dc == 0:
                continue
            a = idx[max(0, -dr):h - max(0, dr), max(0, -dc):w - max(0, dc)]
            b = idx[max(0, dr):h - max(0, -dr), max(0, dc):w - max(0, -dc)]
            rows.append(a.ravel()); cols.append(b.ravel())
    rows, cols = np.concatenate(rows), np.concatenate(cols)
    return coo_matrix((np.ones(len(rows), dtype=int), (rows, cols)), shape=(h * w,) * 2).tocsr()

def cluster_permutation_test_mne(
        subject_maps,
        n_permutations,
        adjacency,
        cluster_alpha=0.01,
        significance_alpha=0.05,
        n_jobs=-1):
    """
    Two-sided one-sample cluster permutation test using MNE.

    Parameters
    ----------
    subject_maps : ndarray
        Shape:
        subjects x height x width

    n_permutations : int
        Number of participant-level sign-flipping permutations.

    cluster_alpha : float
        Two-sided uncorrected alpha used to calculate the
        cluster-forming t threshold.

    significance_alpha : float
        Corrected cluster-level significance threshold.

    n_jobs : int
        Number of parallel CPU jobs. Use -1 for all available cores.

    Returns
    -------
    results : dict

    """
    subject_maps = np.asarray(subject_maps, dtype=float, )

    number_of_subjects = subject_maps.shape[0]

    if number_of_subjects < 2:
        raise ValueError("At least two participants are required.")

    # MNE should not receive NaN or infinite values.
    if not np.all(np.isfinite(subject_maps)):
        raise ValueError(
            "MNE input contains NaN or infinite values. "
            "Remove participants with missing maps or handle "
            "missing values before running the test.")

    degrees_of_freedom = number_of_subjects - 1

    # Two-sided cluster-forming threshold.
    # Pixels with |t| above this are candidates for clustering. This is a cluster-FORMING
    # threshold only; it is not the significance criterion.
    t_threshold = t_distribution.ppf(1 - cluster_alpha / 2, df=degrees_of_freedom, )

    # tail=0 -> two-sided; positive and negative clusters are formed separately.
    # cluster_p_values are already corrected for multiple comparisons across pixels, by
    # comparing each cluster's mass to the null distribution of the maximum cluster mass
    # across permutations.
    (
        observed_t_map,
        clusters,
        cluster_p_values,
        null_distribution,
    ) = permutation_cluster_1samp_test(
        X=subject_maps,
        threshold=t_threshold,
        n_permutations=n_permutations,
        tail=0,
        stat_fun=safe_t_statistic,
        adjacency=adjacency,
        out_type="mask",
        n_jobs=n_jobs, )

    positive_significant_mask = np.zeros(observed_t_map.shape, dtype=bool, )

    negative_significant_mask = np.zeros(observed_t_map.shape, dtype=bool, )

    significant_clusters = []

    for cluster_number, (cluster_mask, corrected_p,) in enumerate(zip(clusters, cluster_p_values), start=1, ):
        cluster_mask = np.asarray(cluster_mask, dtype=bool, )

        cluster_t_values = observed_t_map[cluster_mask]

        # Positive and negative clusters are formed separately.
        cluster_sign = (
            1
            if np.nanmean(cluster_t_values) > 0
            else -1)

        # Cluster mass = sum of |t| over the cluster's pixels.
        cluster_mass = np.sum(np.abs(cluster_t_values))

        cluster_result = {
            "number": cluster_number,
            "sign": int(cluster_sign),
            "size": int(np.sum(cluster_mask)),
            "mass": float(cluster_mass),
            "p_value": float(corrected_p),
            "mask": cluster_mask, }

        # NOTE: significance_alpha is Bonferroni-corrected across emotions in the main script
        # (0.05 / 16), on top of MNE's within-map cluster correction.
        if corrected_p <= significance_alpha:
            if cluster_sign > 0:
                positive_significant_mask |= (cluster_mask)
            else:
                negative_significant_mask |= (cluster_mask)

            significant_clusters.append(cluster_result)

    return {
        "t_map": observed_t_map,
        "positive_significant_mask":
            positive_significant_mask,
        "negative_significant_mask":
            negative_significant_mask,
        "significant_clusters":
            significant_clusters, }


def analyze_all_emotions_mne(
        labels,
        n_permutations,
        adjacency,
        npz_folder,
        output_folder,
        results_path,
        height,
        width,
        cluster_alpha=0.01,
        significance_alpha=0.05,
        n_jobs=-1, ):
    """
    Run a separate MNE one-sample cluster permutation test for every
    emotion, loading only that emotion's maps (all subjects) from disk
    at a time rather than holding every subject x every emotion in
    memory at once.
    """
    results_by_emotion = {}
    average_maps = {}

    for emotion_index, emotion_name in enumerate(labels):
        print(f"\n{'=' * 70}\nMNE emotion test {emotion_index + 1}/{len(labels)}: {emotion_name}\n{'=' * 70}")

        subject_maps = load_one_emotion(npz_folder, emotion_index,height,width,)

        emotion_results = cluster_permutation_test_mne(
            subject_maps=subject_maps,
            n_permutations=n_permutations,
            adjacency=adjacency,
            cluster_alpha=cluster_alpha,
            significance_alpha=significance_alpha,
            n_jobs=n_jobs, )

        results_by_emotion[emotion_name] = emotion_results

        print(
            f"{emotion_name}: "
            f"{len(emotion_results['significant_clusters'])} "
            "significant clusters"
        )

        for cluster in emotion_results["significant_clusters"]:
            direction = (
                "positive activation"
                if cluster["sign"] > 0
                else "negative activation")

            print(
                f"  Cluster {cluster['number']}: "
                f"{direction}, "
                f"size={cluster['size']} pixels, "
                f"mass={cluster['mass']:.2f}, "
                f"corrected p={cluster['p_value']:.4f}")

        # Needed for the group-average plot later; computed now while
        # subject_maps is still in memory, then it's freed below.
        average_maps[emotion_name] = np.nanmean(subject_maps, axis=0)

        del subject_maps

        # Only significant clusters are written to the CSV, so an emotion with none adds no rows.
        cluster_rows = []

        for cluster in emotion_results["significant_clusters"]:
            direction = (
                "positive"
                if cluster["sign"] > 0
                else "negative")

            cluster_rows.append({
                "emotion": emotion_name,
                "cluster_number": cluster["number"],
                "direction": direction,
                "sign": cluster["sign"],
                "size_pixels": cluster["size"],
                "cluster_mass": cluster["mass"],
                "corrected_p_value": cluster["p_value"],
                "significance_alpha": significance_alpha, })

        # Appended after each emotion (header written once at the bottom of the script),
        # so partial results survive if the run crashes partway.
        # NOTE: this fieldnames list must match the header list at the bottom exactly.
        with open(results_path, "a", newline="", encoding="utf-8") as csv_file:
            writer = csv.DictWriter(csv_file, fieldnames=[
                "emotion",
                "cluster_number",
                "direction",
                "sign",
                "size_pixels",
                "cluster_mass",
                "corrected_p_value",
                "significance_alpha", ])
            writer.writerows(cluster_rows)

        output_path = output_folder / (f"cluster_analysis_{emotion_name}.npz")

        # NOTE: the key "average_maps" now holds a single emotion's map (the full dict was
        # saved here previously). The plural name is a leftover.
        np.savez_compressed(
            output_path,
            average_maps=average_maps[emotion_name],
            t_map=emotion_results["t_map"],
            positive_significant_mask=emotion_results["positive_significant_mask"],
            negative_significant_mask=emotion_results["negative_significant_mask"], )

    return results_by_emotion, average_maps


def visualize_cluster_results(
        average_maps,
        results_by_emotion,
        base_image_path,
        labels,
        plot_columns=7, ):
    base_image = np.asarray(Image.open(base_image_path).convert("RGB"))

    # Crop the base body image to match the maps' width.
    base_crop = base_image[:, LEFT_X1:LEFT_X2, :]

    # Shared symmetric color limits across all emotions so intensities are comparable.
    maximum = max(np.nanmax(np.abs(average_maps[name])) for name in labels)

    if not np.isfinite(maximum) or maximum == 0:
        maximum = 1

    plot_rows = int(np.ceil(len(labels) / plot_columns))

    fig, axes = plt.subplots(
        plot_rows,
        plot_columns,
        figsize=(3 * plot_columns, 5 * plot_rows),
        constrained_layout=True, )

    axes = np.atleast_1d(axes).ravel()

    heatmap = None

    for emotion_index, emotion_name in enumerate(labels):
        axis = axes[emotion_index]
        results = results_by_emotion[emotion_name]

        axis.imshow(base_crop)

        # Semi-transparent average map drawn over the body image.
        heatmap = axis.imshow(
            average_maps[emotion_name],
            cmap="seismic",
            vmin=-maximum,
            vmax=maximum,
            alpha=0.65,
            interpolation="nearest", )

        # contour on a 0/1 mask with levels=[0.5] draws one outline along the cluster boundary.
        # Skipped when the mask is all zeros (contour would warn about no contour levels).
        if np.any(results["positive_significant_mask"]):
            axis.contour(
                results["positive_significant_mask"].astype(float),
                levels=[0.5],
                colors=["red"],
                linewidths=1.5,
                linestyles="dashed", )

        if np.any(results["negative_significant_mask"]):
            axis.contour(
                results["negative_significant_mask"].astype(float),
                levels=[0.5],
                colors=["blue"],
                linewidths=1.5,
                linestyles="dashed", )

        axis.set_title(
            f"{emotion_name}\n{len(results['significant_clusters'])} significant clusters",
            fontsize=9, )

        axis.axis("off")

    # Hide unused axes in the last row (16 plots in a 3 x 7 grid leaves 5 empty).
    for axis in axes[len(labels):]:
        axis.axis("off")

    # One shared colorbar; it takes space from all 16 axes.
    fig.colorbar(
        heatmap,
        ax=axes[:len(labels)].tolist(),
        shrink=0.7,
        label="Activation Intensity")

    fig.suptitle(
        "Emotion Body Maps with Significant Clusters\n"
        "Red = Activation; Blue = Deactivation",
        fontsize=16, )

    plt.show()

    return fig

Change below

In [ ]:
# Same crop used when creating resmat.
# Python excludes the ending index, so this crop is 235 pixels wide.
# If the crop changes, all of these must change together.

LEFT_X1 = 0
LEFT_X2 = 235
width=LEFT_X2-LEFT_X1
height=600

base_image_path = Path("/Users/bb/Documents/Park lab/Embody/base.png")

npz_folder = Path("/Users/bb/Documents/Park lab/Embody/Test")

result_folder = Path("/Users/bb/Documents/Park lab/Embody/Test")

# Timestamp keeps each run's CSV from overwriting the previous one.
run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

cluster_result_path = result_folder / (f"Cluster analysis {run_timestamp}.csv")

# NOTE: order matters. Each label's position must match its index on the last axis of resmat,
# because load_one_emotion selects by position, not by name.
LABELS = [
    'Neutral',
    'Fear',
    'Anger',
    'Disgust',
    'Sadness',
    'Happiness',
    'Lonely',
    'Anxiety',
    'Love',
    'Depression',
    'Pride',
    'Shame',
    'Jealousy',
    'Stress',
    'Migraine',
    'Nausea',
]

# Keep within 0.05 (controls the size and extent of cluster)
cluster_alpha = 0.01

# Bonferroni correction across the 16 emotion tests (0.05 / 16 ~= 0.0031).
significance_alpha = 0.05 / len(LABELS)

# Number of permutations (first run with ~1000 then more around ~10000)
n_permutations = 1000

Run below

In [ ]:
# Start caffeinate in the background
caffeinate_proc = subprocess.Popen(['caffeinate', '-i'])

# Built once and reused for every emotion (same 600 x 235 grid).
adjacency = create_8_connectivity(height, width)

# "w" mode creates/overwrites the CSV and writes the header once; the per-emotion
# appends inside analyze_all_emotions_mne add the rows.
with open(cluster_result_path, "w", newline="", encoding="utf-8") as csv_file:
    writer = csv.DictWriter(
        csv_file,
        fieldnames=[
            "emotion",
            "cluster_number",
            "direction",
            "sign",
            "size_pixels",
            "cluster_mass",
            "corrected_p_value",
            "significance_alpha", ])

    writer.writeheader()

results_by_emotion, average_maps = analyze_all_emotions_mne(
    labels=LABELS,
    n_permutations=n_permutations,
    adjacency=adjacency,
    npz_folder=npz_folder,
    output_folder=result_folder,
    results_path=cluster_result_path,
    cluster_alpha=cluster_alpha,
    significance_alpha=significance_alpha,
    height=height,
    width=width,
    n_jobs=-1, )

fig = visualize_cluster_results(
    average_maps=average_maps,
    results_by_emotion=results_by_emotion,
    base_image_path=base_image_path,
    labels=LABELS, )

caffeinate_proc.terminate()